In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import(
    mean_squared_error,
    mean_absolute_percentage_error,
    mean_absolute_error,
    r2_score
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
import mlflow
import mlflow.tensorflow

: 

In [2]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [3]:
#creer une experience
mlflow.set_experiment("Prediction KCL")

<Experiment: artifact_location='file:c:/Users/Pc/Documents/pfa_stage_project/notebooks/forecasting/model/mlruns/1', creation_time=1786008115758, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786008115758, lifecycle_stage='active', name='Prediction KCL', tags={}, trace_location=None, workspace='default'>

In [ ]:
df=pd.read_csv(r"C:\Users\Pc\Documents\pfa_stage_project\data\processed\model_data\final_dataset.csv")
df.head()

In [ ]:
df.describe()

In [ ]:
df["Date"]=pd.to_datetime(df["Date"])
df=df.sort_values("Date")
df.head()

In [ ]:
features=[
    "KCL",
    "Phosphate",
    "DAP",
    "Gas_EUR",
    "Brent"
]

data=df[features]

In [ ]:
#gestion des valeurs manquantes
data=data.interpolate()
data=data.dropna()
data.shape

In [ ]:
#creation des sequences
def create_dataset(dataset,time_step=12):
    X,y=[],[]

    for i in range(len(dataset)-time_step):
        X.append(dataset[i:i+time_step])
        y.append(dataset[i+time_step,0])

    return np.array(X),np.array(y)
time_step=12

In [ ]:
#separation
train_size = int(len(data) * 0.8)

train_data = data.iloc[:train_size]
test_data = data.iloc[train_size:]

#scaler
scaler = MinMaxScaler()

#scaler seulement sur le train
train_scaled = scaler.fit_transform(train_data)
#transformer le test
test_scaled = scaler.transform(test_data)

test_input = pd.concat([
    train_data.tail(time_step),
    test_data
])


In [ ]:
X_train, y_train = create_dataset(train_scaled, time_step)

test_scaled = scaler.transform(test_input)

X_test, y_test = create_dataset(test_scaled, time_step)

In [ ]:
print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

In [ ]:
with mlflow.start_run(run_name="LSTM"):
    #model lstm
    model=Sequential()
    model.add(
        LSTM(
            units=64,#64 neuronnes memorisant les relations temporelles
            activation='tanh',
            input_shape=(X_train.shape[1],X_train.shape[2])
            #input_shape vaut (12,5)
        )
    )

    #Dense(32) cette couche apprend les relation supplementaires apres le lstm
    #Dense(1) une seule sortie
    model.add(Dropout(0.2))
    model.add(Dense(32,activation='relu'))
    model.add(Dense(1))

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss", #on surveille la perte sur le jeu de validation
    patience=10, #on attend 10 epochs sans amelioration avant d arreter
    restore_best_weights=True
)

In [ ]:
#compilation
model.compile(
    optimizer='adam',
    loss='mse'
)

In [ ]:
model.summary()

In [ ]:
#enregistrer les parametres
mlflow.log_param("time_step", time_step)
mlflow.log_param("epochs", 100)
mlflow.log_param("batch_size", 16)
mlflow.log_param("units", 64)
mlflow.log_param("dropout", 0.2)
mlflow.log_param("optimizer", "adam")
#Entrainement
history=model.fit(
    X_train,
    y_train,
    epochs=100, #le reseau parcourt les donnees 100 fois
    batch_size=16, #il met a jour les apoids apres chaque groupe de 16 sequences
    validation_data=(X_test,y_test),
    callbacks=[early_stop],
    verbose=1 #affiche la progression
)

In [ ]:
y_pred=model.predict(X_test)
print(y_pred.shape)

In [ ]:
#on cree un tableau de 5 colonnes dont la premiere est kcl predite et les autres remplis de 0
pred_temp = np.zeros((len(y_pred), len(features)))

pred_temp[:,0] = y_pred.flatten() #car le scaller a ete entraine sur 5 colonnes

In [ ]:
pred_inverse = scaler.inverse_transform(pred_temp)

y_pred_final = pred_inverse[:,0]
#y_pred_final est y_pred mais denormalisée(remis a l echelle d origine)

In [ ]:
y_test_temp = np.zeros((len(y_test), len(features)))

y_test_temp[:,0] = y_test #on fait la meme chose mais cette fois avec y_test

In [ ]:
real_inverse = scaler.inverse_transform(y_test_temp)

y_test_final = real_inverse[:,0]

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test_final, y_pred_final))

mae = mean_absolute_error(y_test_final, y_pred_final)

mape = mean_absolute_percentage_error(y_test_final, y_pred_final)

r2 = r2_score(y_test_final, y_pred_final)

#sauvegarder les metrics
mlflow.log_metric("RMSE", rmse)
mlflow.log_metric("MAE", mae)
mlflow.log_metric("MAPE", mape)
mlflow.log_metric("R2", r2)

print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"MAPE : {mape*100:.2f}%")
print(f"R²   : {r2:.4f}")

In [ ]:
#sauvegarder le model
mlflow.tensorflow.log_model(
    model,
    artifact_path="lstm_model"
)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(y_test_final,label="Réel")
plt.plot(y_pred_final,label="Prédit")
plt.legend()
plt.savefig("prediction.png")
plt.show()

mlflow.log_artifact("prediction.png")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Courbes d'apprentissage du modèle LSTM")
plt.legend()
plt.grid(True)

plt.savefig("loss.png")
plt.show()


mlflow.log_artifact("loss.png")